# 04 - Integration, features and splits

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-USERNAME/demand-forecasting-capstone/blob/main/notebooks/04_integration_and_features.ipynb)

**Goal:** show how the datasets fit together, how model inputs (features) are built without peeking into the future, and how time is split.



How to run: `Runtime -> Run all` (Colab) or run the cells from top to bottom. The default dataset is `synthetic`, which needs no downloads.

In [ ]:
# ---- Setup: works in Google Colab and on your own computer. Nothing to upload. ----
REPO_URL = "https://github.com/YOUR-USERNAME/demand-forecasting-capstone.git"   # <- set once, the same in every notebook
BRANCH = "main"
USE_DRIVE = False   # True = use a copy of the project folder in your Google Drive instead of GitHub

import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if not os.path.exists("src/demand_agents"):
    if IN_COLAB and USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.chdir("/content/drive/MyDrive/demand-forecasting-capstone")
    elif IN_COLAB:
        if not os.path.exists("/content/project"):
            subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, "/content/project"], check=True)
        os.chdir("/content/project")
    elif os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")                                   # running locally from the notebooks folder
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[colab]"], check=True)
sys.path.insert(0, os.path.abspath("src"))
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 120)
from IPython.display import display, Image, Markdown
print("Project folder:", os.getcwd())

In [ ]:
DATASET = "synthetic"    # "synthetic" | "synthetic_calm" | "m5" | "m5_christmas" | "walmart" | "walmart_synthetic"
from demand_agents.config import load_named
cfg = load_named(DATASET)
print("dataset:", DATASET, "| source:", cfg["data"]["source"], "| frequency:", cfg["data"]["frequency"])

## How the datasets fit together

| Combination | Decision | Why |
|---|---|---|
| M5 sales + calendar + prices | **one table** | same items, stores and days |
| M5 + NOAA weather | **merged by state and date** | weather is an outside factor for the same days; M5 hides store locations, so state-level weather is the best link |
| Walmart + M5 | **not merged** | different stores, period (2010-12 vs 2011-16), measure (weekly dollars vs daily units) and no shared keys. Walmart is a second case study. |
| Walmart + NOAA | **not needed** | Walmart already contains temperature, fuel price, CPI and unemployment |

In [ ]:
from demand_agents.data.loaders import load_dataset
df = load_dataset(cfg)
cov = df.groupby("store_id").agg(rows=("sales", "size"), first=("date", "min"), last=("date", "max"),
                                 temperature_available=("temp", lambda s: f"{s.notna().mean():.0%}")).head(12)
display(cov)

## Features (model inputs)

A forecast is made **`horizon` periods ahead**. So any input built from past sales must be at least that old, otherwise we would be cheating by using information we would not have had.

In [ ]:
from demand_agents.data.features import build_features
fc = cfg["forecast"]
feats, names = build_features(df, fc["horizon"], fc["lags"], fc["rolling_windows"], freq=cfg["data"]["frequency"],
                              extra_features=cfg["data"].get("extra_features", []), price_window=fc.get("price_window", 56))
groups = {"past sales": [n for n in names if n.startswith(("lag_", "roll_"))],
          "calendar": [n for n in names if n in ("dow", "month", "day_of_month", "week_of_year", "is_event")],
          "price": [n for n in names if "price" in n], "weather": [n for n in names if n.startswith("temp")],
          "identifiers": [n for n in names if n.endswith("_code")], "other": list(cfg["data"].get("extra_features", []))}
display(pd.DataFrame([(k, len(v), ", ".join(v)) for k, v in groups.items() if v], columns=["group", "count", "features"]))

## Proof that there is no leakage

We change all sales in the last 30 periods to absurd values. Features of earlier dates (up to `horizon` periods past the change) must not move.

In [ ]:
import numpy as np
cut = df["date"].max() - pd.Timedelta(days=30 if cfg["data"]["frequency"] == "D" else 30 * 7)
step = pd.Timedelta(days=fc["horizon"] if cfg["data"]["frequency"] == "D" else fc["horizon"] * 7)
changed = df.copy(); changed.loc[changed["date"] >= cut, "sales"] = changed["sales"] * 0 + 99999
alt, _ = build_features(changed, fc["horizon"], fc["lags"], fc["rolling_windows"], freq=cfg["data"]["frequency"],
                        extra_features=cfg["data"].get("extra_features", []), price_window=fc.get("price_window", 56))
cols = groups["past sales"]; safe = feats["date"] < cut + step
same = np.allclose(feats.loc[safe, cols].fillna(-1).to_numpy(), alt.loc[safe, cols].fillna(-1).to_numpy())
print("features before the change point are identical:", same, "->", "NO LEAKAGE" if same else "LEAKAGE!")

## Time split

Data is split **in time order**, never randomly. `train` fits the model, `val_cal` calibrates the uncertainty, `val_check` is a held-out window the Critic uses to audit it, and `test` is the final exam, looked at once.

In [ ]:
from demand_agents.data.splits import make_splits, assign_split
sp = make_splits(df["date"], cfg["split"]["val_days"], cfg["split"]["test_days"], cfg["uncertainty"]["check_days"])
split = assign_split(df["date"], sp)
tab = df.assign(split=split).groupby("split")["date"].agg(["min", "max", "nunique"]).loc[["train", "val_cal", "val_check", "test"]]
display(tab.rename(columns={"nunique": "periods"}))
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 1.6))
for i, (name, r) in enumerate(tab.iterrows()):
    ax.barh(0, (r["max"] - r["min"]).days + 1, left=r["min"], label=name)
ax.set_yticks([]); ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, 1.5)); plt.show()

**Next:** `05_agents_and_modelling.ipynb`.